# Notebook 25 — Paper figures

CPU-only, deterministic. Reads exclusively committed CSVs; writes publication figures (PDF + 300-dpi PNG) to results/saber/figures_paper/. Complements the committed pack figures A–H.

In [ ]:
# NB25: paper figures. CPU-only, deterministic, reads ONLY committed CSVs.
# Outputs PDF+PNG (300 dpi) to results/saber/figures_paper/.
from google.colab import drive; drive.mount("/content/drive", force_remount=False)
import json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
R = REPO / "results/saber"
OUT = R / "figures_paper"; OUT.mkdir(parents=True, exist_ok=True)

METHODS = ["random", "magnitude", "taylor", "fisher", "saber_v2"]
COLORS = {"random": "#8c8c8c", "magnitude": "#d95f02", "taylor": "#1b9e77",
          "fisher": "#7570b3", "saber_v2": "#e7298a"}
LABEL = {"random": "Random", "magnitude": "Magnitude", "taylor": "Taylor",
         "fisher": "Fisher", "saber_v2": "V-C"}
plt.rcParams.update({"font.size": 9, "axes.titlesize": 10, "figure.dpi": 110})

def pick(df, *cands):
    for c in cands:
        if c in df.columns: return c
    raise KeyError(f"none of {cands} in {list(df.columns)[:8]}...")

def save(fig, name):
    fig.tight_layout()
    fig.savefig(OUT / f"{name}.pdf")
    fig.savefig(OUT / f"{name}.png", dpi=300)
    print("saved", name)

# ---- F1: pre-recovery preservation (17b canonical registry) ----
reg = pd.read_csv(R / "17b_calibrated_checkpoint_freeze/shallow_frozen_model_registry.csv")
prec = pick(reg, "pre_fine_macro_f1", "pre_macro_f1")
bud = pick(reg, "target_flops", "budget")
fig, ax = plt.subplots(figsize=(5.2, 3.0))
budgets = sorted(reg[bud].unique())
w = 0.15
for i, m in enumerate(METHODS):
    sub = reg[reg["method"] == m].set_index(bud).loc[budgets]
    ax.bar(np.arange(len(budgets)) + (i - 2) * w, sub[prec], width=w,
           color=COLORS[m], label=LABEL[m])
ax.set_xticks(range(len(budgets)))
ax.set_xticklabels([f"{int(b*100)}%" for b in budgets])
ax.set_xlabel("Realised FLOP reduction"); ax.set_ylabel("Fine macro-F1 before recovery")
ax.set_title("Pre-recovery preservation at matched realised cost (shallow)")
ax.legend(ncol=3, fontsize=7.5)
save(fig, "F1_pre_recovery_preservation")

# ---- F2: recovery-realisation variance at depth (NB20 vs 20b, standard) ----
r1 = pd.read_csv(R / "20_depth_probe/depth_regime_results.csv")
r2 = pd.read_csv(R / "20b_depth_checkpoint_freeze/deep_frozen_model_registry.csv")
a1 = r1[r1["regime"] == "standard"].set_index("method")[pick(r1, "awbir")]
a2 = r2[r2["regime"] == "standard"].set_index("method")[pick(r2, "post_awbir", "awbir")]
fig, ax = plt.subplots(figsize=(4.6, 3.0))
for i, m in enumerate(METHODS):
    ax.plot([i, i], [a1[m], a2[m]], color=COLORS[m], lw=1.5, zorder=1)
    ax.scatter([i], [a1[m]], marker="o", color=COLORS[m], zorder=2,
               label="Realisation 1" if i == 0 else None, edgecolor="k", s=45)
    ax.scatter([i], [a2[m]], marker="s", color=COLORS[m], zorder=2,
               label="Realisation 2" if i == 0 else None, edgecolor="k", s=45)
ax.set_xticks(range(len(METHODS))); ax.set_xticklabels([LABEL[m] for m in METHODS])
ax.set_ylabel("AWBIR (validation)")
ax.set_title("Standard recovery at depth: two realisations, identical channels")
ax.legend(fontsize=7.5)
save(fig, "F2_recovery_realisation_variance")

# ---- F3: set-level additivity (NB22) ----
bench = pd.read_csv(R / "22_set_interaction/set_level_causal_benchmark.csv")
rep = pd.read_csv(R / "22_set_interaction/additive_confirmation_report.csv")
sh = pick(bench, "sum_harm_awbir")
rawc = pick(bench, "raw_awbir")
panels = [("deep", 0.40), ("shallow", 0.25), ("shallow", 0.40)]
fig, axes = plt.subplots(1, 3, figsize=(9.0, 2.9), sharey=False)
for ax, (arch, b) in zip(axes, panels):
    cell = bench[(bench["architecture"] == arch) & (np.isclose(bench["budget"], b))]
    disc = cell[cell["split"] == "discovery"]; conf = cell[cell["split"] == "confirmation"]
    row = rep[(rep["architecture"] == arch) & (np.isclose(rep["budget"], b)) &
              (rep["predictor"] == "sum_harm_awbir")].iloc[0]
    xs = np.linspace(cell[sh].min(), cell[sh].max(), 50)
    ax.scatter(disc[sh], disc[rawc], facecolor="none", edgecolor="0.4", s=22,
               label="discovery")
    ax.scatter(conf[sh], conf[rawc], color="#1f77b4", s=22, label="confirmation")
    ax.plot(xs, row["slope"] * xs + row["intercept"], color="0.2", lw=1)
    ax.set_title(f"{arch} @ {int(b*100)}%  (conf ρ={row['confirmation_spearman']:.2f})")
    ax.set_xlabel("Summed one-channel AWBIR harm")
axes[0].set_ylabel("Raw set AWBIR")
axes[0].legend(fontsize=7)
save(fig, "F3_set_additivity")

# ---- F4: recovery reordering (NB22) ----
rec = pd.read_csv(R / "22_set_interaction/minimal_recovery_reordering.csv")
summ = pd.read_csv(R / "22_set_interaction/recovery_reordering_summary.csv")
rr = pick(rec, "raw_awbir"); rv = pick(rec, "recovered_awbir")
fig, axes = plt.subplots(1, 2, figsize=(6.6, 3.0))
for ax, arch in zip(axes, ["shallow", "deep"]):
    cell = rec[rec["architecture"] == arch]
    srow = summ[summ["architecture"] == arch].iloc[0]
    ax.scatter(cell[rr], cell[rv], color="#1f77b4", s=26)
    lim = max(cell[rr].max(), 0.05) * 1.05
    ax.plot([0, lim], [0, lim], color="0.6", lw=0.8, ls="--")
    ax.set_xlim(0, lim); ax.set_ylim(0, lim)
    ax.set_title(f"{arch}: ρ(raw, recovered)={srow['raw_vs_recovered_awbir_spearman']:.2f}")
    ax.set_xlabel("Raw set AWBIR")
axes[0].set_ylabel("Recovered set AWBIR")
fig.suptitle("One minimal epoch decouples recovered risk from raw damage", y=1.02)
save(fig, "F4_recovery_reordering")

# ---- F5: one-shot test audit (NB21) ----
t = pd.read_csv(R / "21_one_shot_test/test_model_summary.csv")
aw = pick(t, "awbir"); b2a = pick(t, "benign_to_attack_rate")
fig, axes = plt.subplots(1, 3, figsize=(9.6, 3.0))
shallow = t[(t["architecture"].str.contains("2block")) & (t["variant"] == "full_recovery")]
for m in METHODS:
    sub = shallow[shallow["method"] == m].sort_values("target_flops")
    axes[0].plot(sub["target_flops"] * 100, sub[aw], marker="o", color=COLORS[m],
                 label=LABEL[m], lw=1.2)
axes[0].set_title("Shallow, full recovery (test)"); axes[0].set_xlabel("Budget (%)")
axes[0].set_ylabel("Test AWBIR"); axes[0].legend(fontsize=6.5)
deep = t[t["architecture"].str.contains("4block") & (t["variant"] != "teacher")]
xpos = {"minimal": 0, "standard": 1}
for m in METHODS:
    sub = deep[deep["method"] == m]
    axes[1].plot([xpos[v] for v in sub["variant"]], sub[aw], marker="o",
                 color=COLORS[m], lw=1.2, label=LABEL[m])
axes[1].set_xticks([0, 1]); axes[1].set_xticklabels(["minimal", "standard"])
axes[1].set_title("Deep @40% (test)"); axes[1].set_ylabel("Test AWBIR")
students = t[t["variant"] != "teacher"]
teach = t[t["variant"] == "teacher"]
axes[2].scatter(np.zeros(len(teach)) + [0, 1][:len(teach)], teach[b2a],
                color="k", marker="D", s=45, label="dense teacher")
axes[2].scatter(np.full(len(students[students["architecture"].str.contains("2block")]), 0)
                + 0.12, students[students["architecture"].str.contains("2block")][b2a],
                color="#1f77b4", s=18, alpha=0.8, label="students")
axes[2].scatter(np.full(len(students[students["architecture"].str.contains("4block")]), 1)
                + 0.12, students[students["architecture"].str.contains("4block")][b2a],
                color="#1f77b4", s=18, alpha=0.8)
axes[2].set_yscale("log"); axes[2].set_xticks([0, 1])
axes[2].set_xticklabels(["shallow", "deep"])
axes[2].set_title("Benign→attack on test (log)"); axes[2].legend(fontsize=6.5)
save(fig, "F5_test_audit")

# ---- F6: regime map of selection-effect magnitude ----
n19 = pd.read_csv(R / "19_recovery_regimes/recovery_regime_combined.csv")
bars, labels = [], []
for regime in ["none", "minimal", "full"]:
    sub = n19[(n19["regime"] == regime) & (np.isclose(n19["budget"], 0.40))]
    bars.append(sub["awbir"].max() - sub["awbir"].min())
    labels.append(f"shallow\n{regime}")
d1 = r1[r1["regime"] == "minimal"]["awbir"]
bars.append(d1.max() - d1.min()); labels.append("deep\nminimal")
bars.append(a1.max() - a1.min()); labels.append("deep\nstd (r1)")
bars.append(a2.max() - a2.min()); labels.append("deep\nstd (r2)")
fig, ax = plt.subplots(figsize=(5.4, 3.0))
ax.bar(range(len(bars)), bars, color=["#88a", "#88a", "#88a", "#a66", "#a66", "#a66"])
ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, fontsize=7.5)
ax.set_ylabel("Across-method AWBIR spread")
ax.set_title("Selection-effect magnitude by regime (validation, 40%)")
save(fig, "F6_regime_map")

print("All figures written to", OUT)


In [ ]:
%%bash
cd /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression
cp ../.gitconfig /root/.gitconfig 2>/dev/null; cp ../.git-credentials /root/.git-credentials 2>/dev/null && chmod 600 /root/.git-credentials
jupyter nbconvert --ClearOutputPreprocessor.enabled=True --inplace notebooks/25_paper_figures.ipynb
git add notebooks/25_paper_figures.ipynb results/saber/figures_paper
git commit -m "NB25: six paper figures (F1 pre-recovery preservation, F2 depth recovery-realisation variance, F3 set additivity, F4 recovery reordering, F5 one-shot test audit, F6 regime map), CPU-only and deterministic from committed CSVs; complements pack figures A-H"
git push origin saber-ids-method